# Class Notes

## 1. Neural Network Optimization and Critical Points

A neural network may stop improving while the loss is still large:

$$
L(\theta_t) \text{ is large}, \qquad \|\nabla_\theta L(\theta_t)\| \approx 0.
$$

Then gradient descent has almost no update:

$$
\theta_{t+1}=\theta_t-\eta\nabla_\theta L(\theta_t) \approx \theta_t.
$$

This is an optimization bottleneck, not necessarily overfitting.

A critical point satisfies

$$
\nabla_\theta L(\theta')=0.
$$

Critical points include local minima, local maxima, and saddle points. A saddle point has both ascent and descent directions near $\theta'$.

## 2. Hessian Matrix

For a twice-differentiable scalar function $f: \mathbb{R}^n \to \mathbb{R}$, the Hessian matrix is defined as

$$
H_f(\mathbf{x}) = \nabla^2 f(\mathbf{x}), \qquad (H_f)_{ij} = \frac{\partial^2 f}{\partial x_i \partial x_j}.
$$

For a loss function $L$, the second-order Taylor approximation around $\theta'$ is

$$
L(\theta) \approx L(\theta') + g^T(\theta-\theta') + \frac{1}{2}(\theta-\theta')^T H(\theta-\theta'),
$$

where

$$
g=\nabla_\theta L(\theta'), \qquad H=\nabla_\theta^2 L(\theta').
$$

At a critical point, $g=0$. Let $v=\theta-\theta'$. Then

$$
L(\theta)-L(\theta') \approx \frac{1}{2}v^T H v.
$$

Let $\lambda_i$ be the eigenvalues of $H$:

- $\lambda_i>0$ for all $i$: local minimum.
- $\lambda_i<0$ for all $i$: local maximum.
- positive and negative eigenvalues coexist: saddle point.

If $Hu=\lambda u$ and $\lambda<0$, then

$$
u^T H u=\lambda\|u\|^2<0.
$$

For small $\epsilon>0$,

$$
L(\theta'+\epsilon u)-L(\theta') \approx \frac{1}{2}\epsilon^2\lambda\|u\|^2<0.
$$

Thus a negative-eigenvalue direction can decrease loss even when the gradient is zero.

## 3. Hessian Examples

If

$$
f(x,y)=3x^2+xy+y^2,
$$

then

$$
H_f=
\begin{bmatrix}
6 & 1 \\
1 & 2
\end{bmatrix}.
$$

Since $H_f\succ0$, $(0,0)$ is a local minimum.

For a tiny network

$$
\hat y=w_1w_2x, \qquad x=1, \qquad y=1,
$$

use

$$
L(w_1,w_2)=(w_1w_2-1)^2.
$$

At $(w_1,w_2)=(0,0)$,

$$
\nabla L(0,0)=0, \qquad
H(0,0)=
\begin{bmatrix}
0 & -2 \\
-2 & 0
\end{bmatrix}.
$$

The eigenvalues are $2$ and $-2$, so the origin is a saddle point.

## 4. High-Dimensional Loss Surfaces

Full Hessian eigendecomposition is usually too expensive for modern deep networks. Still, the Hessian view explains why strict bad local minima are rare in high dimensions.

Define the minimum ratio as

$$
\text{minimum ratio}=\frac{\#\{\lambda_i>0\}}{\#\{\lambda_i\}}.
$$

If the ratio is $1$, all local curvature is positive. If the ratio is below $1$, some negative-curvature directions still exist. In high-dimensional neural networks, stuck high-loss points are often saddle points or flat regions rather than strict local minima.

## 5. Batching and Momentum

For dataset size $N$ and batch size $B$, full-batch training uses

$$
L(\theta)=\frac{1}{N}\sum_{n=1}^N \ell_n(\theta), \qquad B=N.
$$

Mini-batch training uses a stochastic approximation

$$
L_{\mathcal{B}}(\theta)=\frac{1}{B}\sum_{i\in\mathcal{B}}\ell_i(\theta), \qquad B<N.
$$

Different batches induce different local objectives. Even if

$$
\nabla L_{\mathcal{B}_1}(\theta)\approx0,
$$

usually

$$
\nabla L_{\mathcal{B}_2}(\theta)\ne0.
$$

Thus mini-batch noise can move parameters away from stationary or sharp regions. Small batches often favor flatter minima; large batches behave closer to deterministic descent and may enter sharper minima.

Momentum accumulates past gradients:

$$
m_{t+1}=\lambda m_t-\eta g_t, \qquad \theta_{t+1}=\theta_t+m_{t+1}, \qquad g_t=\nabla L(\theta_t).
$$

With $m_0=0$,

$$
m_{t+1}=-\eta\sum_{\tau=0}^t\lambda^{t-\tau}g_\tau.
$$

So the update is an exponentially weighted sum of historical gradients, which smooths noisy directions and preserves consistent motion.

## 6. Adaptive Learning Rate

Large gradients do not always imply effective progress. In narrow valleys, updates may oscillate across steep directions. Use parameter-wise scaling:

$$
\theta_{t+1,i}=\theta_{t,i}-\frac{\eta_t}{\sigma_{t,i}}g_{t,i}.
$$

Adagrad uses all historical squared gradients:

$$
\sigma_{t,i}=\sqrt{\frac{1}{t+1}\sum_{\tau=0}^t g_{\tau,i}^2}.
$$

Large-gradient coordinates get smaller effective steps; small-gradient coordinates get larger effective steps.

RMSProp replaces the full average with an exponential moving average:

$$
\sigma_{t,i}^2=\alpha\sigma_{t-1,i}^2+(1-\alpha)g_{t,i}^2, \qquad 0\le\alpha<1.
$$

This makes the denominator depend more on recent curvature.

Adam combines momentum and RMSProp:

$$
m_t\approx \text{EMA}(g_t), \qquad \sigma_t^2\approx \text{EMA}(g_t^2), \qquad \theta_{t+1}=\theta_t-\eta_t\frac{m_t}{\sigma_t}.
$$

Learning-rate schedules control the global scale $\eta_t$. A common decay is

$$
\eta_t=\frac{\eta_0}{\sqrt{t+1}}.
$$

Warmup starts with small $\eta_t$, then increases it before decay, so early unstable gradient statistics have limited effect.

## 7. Classification Loss

For $K$-class classification, encode labels by one-hot vectors

$$
\hat y\in\{e_1,\dots,e_K\}.
$$

Given logits $z\in\mathbb{R}^K$, softmax gives

$$
p_i=\frac{e^{z_i}}{\sum_{j=1}^K e^{z_j}}, \qquad p_i\in(0,1), \qquad \sum_i p_i=1.
$$

Two common losses are

$$
L_{\mathrm{MSE}}=\sum_{i=1}^K(p_i-\hat y_i)^2,
$$

$$
L_{\mathrm{CE}}=-\sum_{i=1}^K\hat y_i\log p_i.
$$

If the true class is $c$, then

$$
L_{\mathrm{CE}}=-\log p_c.
$$

For softmax with cross entropy,

$$
\frac{\partial L_{\mathrm{CE}}}{\partial z_i}=p_i-\hat y_i.
$$

When the model is confidently wrong, cross entropy keeps a large useful gradient. MSE after softmax can create flatter regions, making optimization harder. Thus cross entropy is usually preferred for classification.

## 8. Batch Normalization

If feature scales differ greatly, a linear model

$$
y=w_1x_1+w_2x_2+b
$$

has uneven sensitivity:

$$
\Delta y=\Delta w_i x_i.
$$

Large-scale inputs create steep directions; small-scale inputs create flat directions. This stretches the loss contour and makes optimization unstable.

For activations $z^1,\dots,z^B$ in one batch,

$$
\mu_{\mathcal{B}}=\frac{1}{B}\sum_{b=1}^B z^b, \qquad
\sigma_{\mathcal{B}}^2=\frac{1}{B}\sum_{b=1}^B(z^b-\mu_{\mathcal{B}})^2.
$$

Normalize and restore learnable scale:

$$
\tilde z^b=\frac{z^b-\mu_{\mathcal{B}}}{\sqrt{\sigma_{\mathcal{B}}^2+\epsilon}}, \qquad
\hat z^b=\gamma\tilde z^b+\beta.
$$

Here $\gamma$ and $\beta$ are learned parameters.

During inference, use running statistics from training:

$$
\bar\mu\leftarrow(1-p)\bar\mu+p\mu_{\mathcal{B}}, \qquad
\bar\sigma^2\leftarrow(1-p)\bar\sigma^2+p\sigma_{\mathcal{B}}^2.
$$

Batch normalization is useful mainly because it smooths the optimization surface and permits larger learning rates, not merely because it stabilizes hidden-layer distributions.

## 9. Convolutional Neural Networks

An image is a tensor

$$
X\in\mathbb{R}^{H\times W\times C}.
$$

Flattening gives $x\in\mathbb{R}^D$, where $D=HWC$. A fully connected layer with $M$ hidden units has

$$
MD+M
$$

parameters, so high-resolution images easily create very large models.

CNNs add image-specific bias:

- local receptive fields: each neuron uses a patch of size $k_h\times k_w\times C_{\mathrm{in}}$;
- weight sharing: the same filter is applied at different spatial locations.

For input $X\in\mathbb{R}^{H\times W\times C_{\mathrm{in}}}$ and filters $W\in\mathbb{R}^{k\times k\times C_{\mathrm{in}}\times C_{\mathrm{out}}}$,

$$
Y_{i,j,c}=\sigma\left(\sum_{m=1}^{k}\sum_{n=1}^{k}\sum_{d=1}^{C_{\mathrm{in}}}X_{is+m,js+n,d}W_{m,n,d,c}+b_c\right).
$$

The output $Y$ is a feature map. Stacking small kernels increases the effective receptive field while keeping parameter count small.

Pooling is a fixed downsampling operator. For max pooling,

$$
Y_{i,j,c}=\max_{(m,n)\in\Omega_{i,j}}X_{m,n,c}.
$$

It reduces spatial resolution and computation, but may discard local details.

## 10. Neighbor Embedding

Manifold learning assumes high-dimensional samples $x_i\in\mathbb{R}^D$ lie near a low-dimensional manifold. The goal is an embedding

$$
x_i\mapsto z_i\in\mathbb{R}^d, \qquad d\ll D,
$$

that preserves local neighborhood geometry.

Locally Linear Embedding first reconstructs each point by its neighbors $N(i)$:

$$
\min_W\sum_i\left\|x_i-\sum_{j\in N(i)}w_{ij}x_j\right\|^2,
\qquad \sum_{j\in N(i)}w_{ij}=1.
$$

Then it finds low-dimensional points with the same local weights:

$$
\min_Z\sum_i\left\|z_i-\sum_{j\in N(i)}w_{ij}z_j\right\|^2.
$$

LLE preserves local reconstruction, but does not explicitly push non-neighbor points apart.

Laplacian Eigenmaps build a weighted graph $W=(w_{ij})$ and minimize

$$
\frac{1}{2}\sum_{i,j}w_{ij}\|z_i-z_j\|^2=\operatorname{tr}(Z^TLZ),
\qquad L=D-W,
$$

with constraint

$$
Z^TDZ=I, \qquad D_{ii}=\sum_jw_{ij}.
$$

The solution is given by the smallest non-trivial generalized eigenvectors of

$$
Lv=\lambda Dv.
$$

t-SNE converts pairwise similarities into probability distributions. In the original space,

$$
p_{j|i}=\frac{\exp(-\|x_i-x_j\|^2/2\sigma_i^2)}{\sum_{k\ne i}\exp(-\|x_i-x_k\|^2/2\sigma_i^2)},
\qquad p_{ij}=\frac{p_{j|i}+p_{i|j}}{2N}.
$$

In the embedding space, use a heavy-tailed Student distribution:

$$
q_{ij}=\frac{(1+\|z_i-z_j\|^2)^{-1}}{\sum_k\sum_{m\ne k}(1+\|z_k-z_m\|^2)^{-1}}.
$$

Optimize

$$
\mathrm{KL}(P\|Q)=\sum_i\sum_jp_{ij}\log\frac{p_{ij}}{q_{ij}}.
$$

The heavy tail lets moderately distant embedded points remain probable, which helps separate clusters and reduces crowding.